# PlantDoc + YOLOv8 - protocolo experimental do TCC

Este notebook executa o protocolo em fases e salva checkpoints no Google Drive. Antes de começar, selecione **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**. T4 é suficiente para YOLOv8n; A100 reduz o tempo.

As fases podem ser executadas em sessões diferentes. Ao repetir uma célula, runs concluídos são detectados e não são treinados novamente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO = Path('/content/plantdoc-yolov8-tcc')
if not REPO.exists():
    !git clone https://github.com/SoulStorm0/plantdoc-yolov8-tcc.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd /content/plantdoc-yolov8-tcc
!python -m pip install -q -e .

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU não detectada. Ative uma GPU no ambiente de execução do Colab.')
DEVICE = '0'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. Preparação e auditoria do PlantDoc

O conversor lê a fonte oficial em Pascal VOC, cria nomes seguros, remove duas classes sem suporte estatístico mínimo e produz 27 classes com split determinístico 70/20/10. O conjunto de teste permanece isolado durante toda a seleção.

In [ ]:
SOURCE = Path('/content/plantdoc_official')
DATASET = Path('/content/plantdoc_yolo_27')
if not SOURCE.exists():
    !git clone --depth 1 --no-checkout https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git {SOURCE}
if not (DATASET / 'data.yaml').exists():
    !python scripts/prepare_official_plantdoc.py --repo {SOURCE} --output {DATASET} --min-class-instances 20
DATA_YAML = DATASET / 'data.yaml'
!python -m plantdoc_tcc audit --data {DATA_YAML} --split train --expected-classes 27
!python -m plantdoc_tcc audit --data {DATA_YAML} --split val --expected-classes 27
!python -m plantdoc_tcc audit --data {DATA_YAML} --split test --expected-classes 27

In [ ]:
RUN_ROOT = Path('/content/drive/MyDrive/TCC_PlantDoc/runs_staged')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROTOCOL = REPO / 'configs/colab_protocol.json'
print('Checkpoints e resultados:', RUN_ROOT)
print('Protocolo:', PROTOCOL)

## 2. Teste rápido de uma época (execute antes de assinar ou iniciar as fases)

Esta célula treina apenas o `baseline` por **uma época**, usando 5% do treino e imagens 320x320. Ela verifica GPU, leitura do dataset, treinamento e salvamento no Drive. O resultado é somente técnico: **não deve ser usado nas tabelas do TCC**. Execute as células anteriores e esta célula; não é necessário iniciar a Fase A para fazer o teste.

In [ ]:
import json
SANITY_ROOT = Path('/content/drive/MyDrive/TCC_PlantDoc/sanity_one_epoch')
SANITY_ROOT.mkdir(parents=True, exist_ok=True)
sanity = json.loads((REPO / 'configs/integration.json').read_text(encoding='utf-8'))
sanity['project'] = str(SANITY_ROOT)
SANITY_CONFIG = Path('/content/plantdoc_sanity_one_epoch.json')
SANITY_CONFIG.write_text(json.dumps(sanity, indent=2), encoding='utf-8')
!python -m plantdoc_tcc train --data {DATA_YAML} --config {SANITY_CONFIG} --strategy baseline --epoch 1 --device {DEVICE}
checkpoints = list(SANITY_ROOT.glob('*/weights/best.pt'))
if not checkpoints:
    raise RuntimeError('O teste terminou sem gerar weights/best.pt.')
print('Teste concluído com sucesso. Checkpoint:', checkpoints[-1])

## 3. Teste de uma época com o PlantDoc completo

Execute esta célula para medir o funcionamento real com **100% do treino**, validação completa, imagens 640x640 e batch 16. Ela treina somente o `baseline` por uma época, registra o tempo e salva checkpoint e métricas no Drive. Ainda é um teste técnico e não substitui as fases científicas.

In [ ]:
import json
import time
import pandas as pd
FULL_CHECK_ROOT = Path('/content/drive/MyDrive/TCC_PlantDoc/full_dataset_one_epoch')
FULL_CHECK_ROOT.mkdir(parents=True, exist_ok=True)
full_check = json.loads((REPO / 'configs/integration.json').read_text(encoding='utf-8'))
full_check.update({
    'project': str(FULL_CHECK_ROOT),
    'imgsz': 640,
    'fraction': 1.0,
    'batch': [16],
    'workers': 2,
})
FULL_CHECK_CONFIG = Path('/content/plantdoc_full_one_epoch.json')
FULL_CHECK_CONFIG.write_text(json.dumps(full_check, indent=2), encoding='utf-8')
started = time.perf_counter()
!python -m plantdoc_tcc train --data {DATA_YAML} --config {FULL_CHECK_CONFIG} --strategy baseline --epoch 1 --device {DEVICE}
elapsed_minutes = (time.perf_counter() - started) / 60
result_files = list(FULL_CHECK_ROOT.glob('*/results.csv'))
checkpoints = list(FULL_CHECK_ROOT.glob('*/weights/best.pt'))
if not result_files or not checkpoints:
    raise RuntimeError('A época completa terminou sem gerar métricas ou weights/best.pt.')
latest_results = max(result_files, key=lambda path: path.stat().st_mtime)
latest_checkpoint = max(checkpoints, key=lambda path: path.stat().st_mtime)
metrics = pd.read_csv(latest_results).iloc[-1]
print(f'Teste completo concluído em {elapsed_minutes:.1f} minutos.')
print('Checkpoint:', latest_checkpoint)
display(metrics.to_frame('valor'))

## 4. Fase A - comparação das funções de custo

Treina `baseline`, ponderação por classe e Focal Loss por 100 épocas com os mesmos hiperparâmetros. Execute um índice por sessão: **1, 2 e 3**. Depois do terceiro, a melhor estratégia é escolhida por mAP@50:95 na validação.

In [ ]:
LOSS_RUN_INDEX = 1  # altere para 1, depois 2, depois 3
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase loss --run-index {LOSS_RUN_INDEX} --device {DEVICE}
!python -m plantdoc_tcc status --config {PROTOCOL} --project {RUN_ROOT}

## 5. Fase B - busca de hiperparâmetros

Executa oito combinações planejadas por 100 épocas usando somente a loss vencedora. Execute um índice por sessão, de **1 a 8**. O espaço cobre batch 16/32, learning rate de 1e-3 a 1e-5, momentum 0,9/0,937 e weight decay de 1e-4 a 1e-3, com warmup e cosine annealing.

In [ ]:
SEARCH_RUN_INDEX = 1  # altere de 1 ate 8, seguindo pending_indices
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase search --run-index {SEARCH_RUN_INDEX} --device {DEVICE}
!python -m plantdoc_tcc status --config {PROTOCOL} --project {RUN_ROOT}

## 6. Fase C - promoção para 200 épocas

As três melhores combinações da validação são treinadas novamente por 200 épocas. Execute separadamente os índices **1, 2 e 3**.

In [ ]:
PROMOTION_RUN_INDEX = 1  # altere para 1, depois 2, depois 3
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase promote200 --run-index {PROMOTION_RUN_INDEX} --device {DEVICE}
!python -m plantdoc_tcc status --config {PROTOCOL} --project {RUN_ROOT}

## 7. Fase D - confirmação em 300 épocas

A melhor configuração de 200 épocas é confirmada em 300 épocas. Ainda é usada somente a validação.

In [ ]:
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase confirm300 --run-index 1 --device {DEVICE}
!python -m plantdoc_tcc status --config {PROTOCOL} --project {RUN_ROOT}

## 8. Backup opcional dos checkpoints no Kaggle

O Google Drive já é o armazenamento principal. Esta etapa cria ou atualiza um **dataset privado** no Kaggle contendo um ZIP de `runs_staged`. No Colab, abra o painel de Secrets (ícone de chave), crie `KAGGLE_API_TOKEN` e permita o acesso ao notebook. Informe abaixo apenas o seu nome de usuário público do Kaggle; nunca escreva o token diretamente na célula. Execute o backup depois de cada run concluído.

In [ ]:
ENABLE_KAGGLE_BACKUP = False
KAGGLE_USERNAME = 'SEU_USUARIO_KAGGLE'
KAGGLE_DATASET_SLUG = 'plantdoc-yolov8-tcc-checkpoints'

if not ENABLE_KAGGLE_BACKUP:
    print('Backup Kaggle desativado. Altere ENABLE_KAGGLE_BACKUP para True quando desejar enviar.')
else:
    import json, os, shutil, subprocess, sys
    from google.colab import userdata
    if KAGGLE_USERNAME == 'SEU_USUARIO_KAGGLE':
        raise ValueError('Preencha KAGGLE_USERNAME com seu usuário público do Kaggle.')
    try:
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    except Exception as exc:
        raise RuntimeError('Adicione KAGGLE_API_TOKEN aos Secrets do Colab.') from exc
    if not kaggle_token:
        raise RuntimeError('O Secret KAGGLE_API_TOKEN está vazio ou sem permissão.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    upload_dir = Path('/content/kaggle_checkpoint_upload')
    if upload_dir.exists():
        shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True)
    archive = shutil.make_archive(
        str(upload_dir / 'runs_staged'), 'zip', root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name
    )
    dataset_handle = f'{KAGGLE_USERNAME}/{KAGGLE_DATASET_SLUG}'
    metadata = {
        'title': 'PlantDoc YOLOv8 TCC Checkpoints',
        'id': dataset_handle,
        'licenses': [{'name': 'other'}],
    }
    (upload_dir / 'dataset-metadata.json').write_text(
        json.dumps(metadata, indent=2), encoding='utf-8'
    )
    env = os.environ.copy()
    env['KAGGLE_API_TOKEN'] = kaggle_token
    status = subprocess.run(
        ['kaggle', 'datasets', 'status', dataset_handle], env=env, capture_output=True, text=True
    )
    if status.returncode == 0:
        command = ['kaggle', 'datasets', 'version', '-p', str(upload_dir), '-m', 'Atualização dos checkpoints do TCC', '-q']
    else:
        command = ['kaggle', 'datasets', 'create', '-p', str(upload_dir), '-q']
    subprocess.run(command, check=True, env=env)
    print('Backup privado enviado:', dataset_handle)
    print('Arquivo:', archive)

## 9. Avaliação final no teste - execute uma única vez

Esta etapa calcula as métricas finais somente depois da seleção completa. O pipeline grava uma trava no resumo e bloqueia uma segunda avaliação, evitando ajuste indireto ao teste. Altere `CONFIRM_FINAL_TEST` para `True` apenas quando a Fase D estiver concluída.

In [ ]:
CONFIRM_FINAL_TEST = False
if not CONFIRM_FINAL_TEST:
    raise RuntimeError('Confirmação necessária: altere CONFIRM_FINAL_TEST para True.')
!python -m plantdoc_tcc final-test --data {DATA_YAML} --project {RUN_ROOT}

## 10. Resumo para o TCC

`protocol_runs.csv` contém todos os experimentos; `protocol_summary.json` registra a loss vencedora, o melhor modelo e a avaliação final; `final_test_metrics.json` contém métricas globais e por classe. Todos permanecem no Google Drive.

In [ ]:
import json
import pandas as pd
display(pd.read_csv(RUN_ROOT / 'protocol_runs.csv').sort_values('map50_95', ascending=False))
summary = json.loads((RUN_ROOT / 'protocol_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))